In [ ]:
# Ejemplo de red bayesiana
# Simulación para determinar la probabilidad de aprobación de un estudiante
# considerando sus horas de estudio, asistencia y nota del examen.

#        | Horas de estudio |
#                 |
#                 v
#          | Asistencia |
#                 |
#                 v
#          | Nota examen |
#                 |
#                 v
#          | Aprobación |

In [ ]:
!pip install pgmpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 6.7 MB/s eta 0:00:00


In [ ]:
# librería de redes bayesianas
# crear la red bayesiana
from pgmpy.models import DiscreteBayesianNetwork

# crear tablas de probabilidades condicionales
from pgmpy.factors.discrete import TabularCPD

# realizar inferencia en la red bayesiana
from pgmpy.inference import VariableElimination

/usr/local/lib/python3.13/dist-packages/pgmpy/estimators/__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


In [ ]:
# crear la red bayesiana

model = DiscreteBayesianNetwork([
    ('horas_estudio', 'asistencia'),
    ('asistencia', 'nota_examen'),
    ('nota_examen', 'aprobacion')
])

In [ ]:
# crear la tabla de probabilidades para la variable horas_estudio

# Supuesto:
# 40% de probabilidad de que un estudiante tenga horas de estudio altas
# 60% de probabilidad de que un estudiante tenga horas de estudio bajas

cpd_horas_estudio = TabularCPD(
    variable='horas_estudio',
    variable_card=2,
    values=[[0.4], [0.6]],
    state_names={
        'horas_estudio': ['Alta', 'Baja']
    }
)

In [ ]:
# crear la tabla de probabilidades para la variable asistencia

# Supuesto:
# Si las horas de estudio son altas:
#   80% de probabilidad de tener asistencia alta
#   20% de probabilidad de tener asistencia baja
#
# Si las horas de estudio son bajas:
#   40% de probabilidad de tener asistencia alta
#   60% de probabilidad de tener asistencia baja

cpd_asistencia = TabularCPD(
    variable='asistencia',
    variable_card=2,
    values=[
        [0.8, 0.4],
        [0.2, 0.6]
    ],
    evidence=['horas_estudio'],
    evidence_card=[2],
    state_names={
        'asistencia': ['Alta', 'Baja'],
        'horas_estudio': ['Alta', 'Baja']
    }
)

In [ ]:
print(cpd_asistencia)

+------------------+---------------------+---------------------+
| horas_estudio    | horas_estudio(Alta) | horas_estudio(Baja) |
+------------------+---------------------+---------------------+
| asistencia(Alta) | 0.8                 | 0.4                 |
+------------------+---------------------+---------------------+
| asistencia(Baja) | 0.2                 | 0.6                 |
+------------------+---------------------+---------------------+


In [ ]:
# crear la tabla de probabilidades condicionales para nota_examen

# Supuesto:
# Si la asistencia es alta:
#   85% de probabilidad de obtener una nota aprobatoria
#   15% de probabilidad de obtener una nota desaprobatoria
#
# Si la asistencia es baja:
#   45% de probabilidad de obtener una nota aprobatoria
#   55% de probabilidad de obtener una nota desaprobatoria

cpd_nota_examen = TabularCPD(
    variable='nota_examen',
    variable_card=2,
    values=[
        [0.85, 0.45],
        [0.15, 0.55]
    ],
    evidence=['asistencia'],
    evidence_card=[2],
    state_names={
        'nota_examen': ['Aprobada', 'Desaprobada'],
        'asistencia': ['Alta', 'Baja']
    }
)

In [ ]:
# crear la tabla de probabilidades condicionales para aprobacion

# Supuesto:
# Si la nota del examen es aprobada:
#   95% de probabilidad de aprobar el curso
#   5% de probabilidad de no aprobar
#
# Si la nota del examen es desaprobada:
#   10% de probabilidad de aprobar el curso
#   90% de probabilidad de no aprobar

cpd_aprobacion = TabularCPD(
    variable='aprobacion',
    variable_card=2,
    values=[
        [0.95, 0.10],
        [0.05, 0.90]
    ],
    evidence=['nota_examen'],
    evidence_card=[2],
    state_names={
        'aprobacion': ['Si', 'No'],
        'nota_examen': ['Aprobada', 'Desaprobada']
    }
)

In [ ]:
# agregar al modelo las tablas de probabilidades condicionales

model.add_cpds(
    cpd_horas_estudio,
    cpd_asistencia,
    cpd_nota_examen,
    cpd_aprobacion
)

# verificar que el modelo esté correctamente construido
model.check_model()

True

In [ ]:
print("Modelo de red bayesiana creado y verificado correctamente")
print(model.edges())

Modelo de red bayesiana creado y verificado correctamente
[('horas_estudio', 'asistencia'), ('asistencia', 'nota_examen'), ('nota_examen', 'aprobacion')]


In [ ]:
# inferencia en la red bayesiana
# motor o proceso de inferencia para el grafo

inferencia = VariableElimination(model)

resultado_aprobacion = inferencia.query(
    variables=['aprobacion']
)

print("Probabilidad de aprobar el curso:")
print(resultado_aprobacion)

Probabilidad de aprobar el curso:
+----------------+-------------------+
| aprobacion     |   phi(aprobacion) |
+================+===================+
| aprobacion(Si) |            0.6729 |
+----------------+-------------------+
| aprobacion(No) |            0.3271 |
+----------------+-------------------+
